In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib tensorflow

In [ ]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

from google.colab import files
from tensorflow.keras.preprocessing.sequence import pad_sequences

RD_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RD_SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(RD_SEED)
np.random.seed(RD_SEED)

tf.keras.utils.set_random_seed(RD_SEED)
tf.config.experimental.enable_op_determinism()

uploaded = files.upload()

TRAIN_FILE = "train_001_v1.csv"
TEST_FILE = "test_002.csv"

train_df_f = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)


required_columns = {
    "transaction_id",
    "sequence_no",
    "sequence_token",
    "label",
    "flow_id"
}

In [ ]:
#############################################################
# E4 – R2: Remove value transitions from sequence tokens

# def remove_value(token):
#     token = str(token)
#     return token.rsplit(":", 1)[0] if ":" in token else token

# train_df_f["sequence_token"] = (
#     train_df_f["sequence_token"].apply(remove_value)
# )

# test_df["sequence_token"] = (
#     test_df["sequence_token"].apply(remove_value)
# )

# print("Training example:")
# print(train_df_f["sequence_token"].iloc[0])

# print("Testing example:")
# print(test_df["sequence_token"].iloc[0])

#################################################################

# E4 – R3: Remove user roles

# def remove_role(token):
#     return str(token).split(":", 1)[1]

# train_df_f["sequence_token"] = (
#     train_df_f["sequence_token"].apply(remove_role)
# )

# test_df["sequence_token"] = (
#     test_df["sequence_token"].apply(remove_role)
# )

#################################################################

# E4 – R4: Remove role and value transition

# def structural_token(token):
#     parts = str(token).split(":")
#     return ":".join(parts[1:-1])

# train_df_f["sequence_token"] = (
#     train_df_f["sequence_token"].apply(structural_token)
# )

# test_df["sequence_token"] = (
#     test_df["sequence_token"].apply(structural_token)
# )


In [ ]:
for name, df in [("training", train_df_f), ("testing", test_df)]:
    missing_cols = required_columns.difference(df.columns)
    if missing_cols:
        raise ValueError(f"{name.title()} dataset missing columns: {sorted(missing_cols)}")
    df["sequence_token"] = df["sequence_token"].fillna("MISSING_TOKEN").astype(str)


def build_transaction_table(df: pd.DataFrame) -> pd.DataFrame:
    ordered = df.copy()
    group_cols = ["transaction_id"]
    if "dataset_id" in ordered.columns:
        ordered["dataset_id"] = ordered["dataset_id"].fillna("UNKNOWN").astype(str)
        group_cols.insert(0, "dataset_id")

    ordered = ordered.sort_values(group_cols + ["sequence_no"], kind="stable")
    for col in ("label", "flow_id"):
        inconsistent = ordered.groupby(group_cols, dropna=False)[col].nunique(dropna=False)
        if (inconsistent > 1).any():
            raise ValueError(f"Inconsistent {col} within transactions")

    aggregation = {
        "tokens": ("sequence_token", list),
        "label": ("label", "first"),
        "flow_id": ("flow_id", "first"),
    }
    if "variant_id" in ordered.columns:
        aggregation["variant_id"] = ("variant_id", "first")

    transaction_table = (
        ordered.groupby(group_cols, sort=False, dropna=False)
        .agg(**aggregation)
        .reset_index()
    )
    transaction_table["length"] = transaction_table["tokens"].str.len()
    return transaction_table


train_transactions = build_transaction_table(train_df_f)
test_transactions = build_transaction_table(test_df)

train_labels = set(train_transactions["label"].str.lower())
if train_labels != {"normal"}:
    raise ValueError(
        "Training dataset MUST contain only normal transactions. "
        f"Found labels: {sorted(train_labels)}"
    )


In [ ]:
PAD_IDX = 0
OOV_IDX = 1


training_tokens = sorted(
    {
        token
        for sequence in train_transactions["tokens"]
        for token in sequence
    }
)


token_to_idx = {
  token : index
  for index, token in enumerate(training_tokens, start=2)
}

index_to_token = {
  index : token
  for token , index in token_to_idx.items()
}

index_to_token[PAD_IDX] = "<PAD>"
index_to_token[OOV_IDX] = "<OOV>"

VOCAB_SIZE_W = len(token_to_idx) + 2

train_token_set_f = set(training_tokens)

test_token_set_c = {
    token
    for sequence in test_transactions["tokens"]
    for token in sequence
}

unseen_test_tokens = sorted(test_token_set_c - train_token_set_f)

print("Vocabulary size including PAD and OOV:" , VOCAB_SIZE_W)
print("Unique training tokens:", len(train_token_set_f))
print("Unique test tokens:" , len(test_token_set_c))
print("Unseen test tokens:", len(unseen_test_tokens))


for token in unseen_test_tokens:
  print(" -", token)

In [ ]:
def encode_sequences(sequence_series , token_map):
  return [
      [token_map.get(token, OOV_IDX) for token in sequence]
      for sequence in sequence_series
  ]

train_encoded_set = encode_sequences(train_transactions["tokens"], token_to_idx)
test_encoded_set = encode_sequences(test_transactions["tokens"], token_to_idx)

MAXIMUM_LEN = max(
    map(len , train_encoded_set)
  )


X_all_f = tf.keras.utils.pad_sequences(
      train_encoded_set,
      maxlen=MAXIMUM_LEN,
      padding="post",
      truncating="post",
      value = PAD_IDX,
  )

X_test_c = tf.keras.utils.pad_sequences(
      test_encoded_set,
      maxlen = MAXIMUM_LEN,
      padding="post",
      truncating="post",
      value=PAD_IDX,
  )

rng = np.random.default_rng(RD_SEED)

indices = np.arange(len(X_all_f))

rng.shuffle(indices)

validation_fraction = 0.20
validation_size = int(round(len(indices) * validation_fraction))


val_indices = indices[:validation_size]
train_indices = indices[validation_size:]

X_train = X_all_f[train_indices]
X_val_f = X_all_f[val_indices]


train_weights_f = (X_train != PAD_IDX).astype("float32")
val_weights = (X_val_f != PAD_IDX).astype("float32")

In [ ]:
tf.keras.backend.clear_session()

#Model lstm

EMBEDDING_DIM = 32
LATENT_DIM = 64
DROPOUT_RATE = 0.20

inputs = tf.keras.Input(shape=(MAXIMUM_LEN,), dtype="int32", name="token_sequence")

embedded = tf.keras.layers.Embedding(
    input_dim=VOCAB_SIZE_W,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="token_embedding"
)(inputs)

encoded = tf.keras.layers.LSTM(
    LATENT_DIM,
    dropout=DROPOUT_RATE,
    name="encoder_lstm"
)(embedded)

repeated = tf.keras.layers.RepeatVector(MAXIMUM_LEN, name="repeat_context")(encoded)

decoded = tf.keras.layers.LSTM(
   LATENT_DIM,
   return_sequences=True,
   dropout=DROPOUT_RATE,
   name = "decoder_lstm"
)(repeated)

outputs = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Dense(VOCAB_SIZE_W, activation="softmax"),
    name="token_probabilities",
)(decoded)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="erp_lstm_autoencoder")

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
)


model.summary()

In [ ]:
callback = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_lstm_autoencoder.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


history = model.fit(
    X_train,
    X_train,
    sample_weight=train_weights_f,
    validation_data=(X_val_f, X_val_f, val_weights),
    epochs=80,
    batch_size=32,
    shuffle=True,
    callbacks=callback,
    verbose=1,
)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], label="Training  loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Masked categorical cross-entropy")
plt.title("LSTM Autoencoder Training history")

plt.tight_layout()

plt.savefig(
    "pic2.png",
    dpi=600,
    bbox_inches="tight"
)

plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def reconstruction_scores(model , X , batch_size=128):

  prob = model.predict(X, batch_size=batch_size,  verbose=0)

  row_indices = np.arange(X.shape[0])[:, None]

  time_indices = np.arange(X.shape[1])[None, :]

  correct_token_probabilities = prob[
      row_indices,
      time_indices,
      X
  ]

  token_losses = -np.log(
      np.clip(correct_token_probabilities, 1e-8, 1.0)
  )
  mask = (X != PAD_IDX).astype("float32")

  sequence_losses = (token_losses * mask).sum(axis=1) / np.maximum(
      mask.sum(axis=1),
      1.0
  )

  return sequence_losses

train_scores = reconstruction_scores(model, X_train)
val_scores = reconstruction_scores(model, X_val_f)
test_scores = reconstruction_scores(model, X_test_c)

print("Training score mean:", train_scores.mean())
print("Validation score mean:", val_scores.mean())
print("Tests score mean:", test_scores.mean())

In [ ]:
THRESHOLD_PERCENTILE = 99.0
threshold = np.percentile(val_scores, THRESHOLD_PERCENTILE)

print(f"Threshold percentile: {THRESHOLD_PERCENTILE}")
print(f"anomaly threshold: {threshold:.6f}")

plt.figure(figsize=(8,5))

plt.hist(val_scores, bins=30, alpha=0.7, label="Normal validation scores")
plt.axvline(threshold, linestyle="--", label=f"{THRESHOLD_PERCENTILE}th percentile = {threshold:.4f}")


plt.xlabel("Reconstruction score")
plt.ylabel("Number of transactions")
plt.title("Normal Validation Score Distribution")

plt.ticklabel_format(style="plain", axis="x")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    "pic1.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
results = test_transactions.copy()

results["anomaly_score"] = test_scores

results["actual_label"] = (
    results["label"].str.lower().eq("anomaly").astype(int)
)

results["predicted_label"] = (
    results["anomaly_score"] > threshold
).astype(int)

label_name = ["normal" , "anomaly"]


print(
    classification_report(
        results["actual_label"],
        results["predicted_label"],
        target_names=label_name,
        digits=4,
        zero_division=0,
    )
)

conmtx = confusion_matrix(
    results["actual_label"],
    results["predicted_label"],
    labels=[0, 1],

)


conmtx_dis = ConfusionMatrixDisplay(
    confusion_matrix=conmtx,
    display_labels=label_name
)


conmtx_dis.plot(values_format="d")
plt.title("Transaction-Level Confusion Matrix")

print("Accuracy:" , accuracy_score(
    results["actual_label"], results["predicted_label"]
))

print("Precision:" , precision_score(
    results["actual_label"], results["predicted_label"], zero_division =0
))
print("Recall :" , recall_score(
    results["actual_label"], results["predicted_label"], zero_division =0
))
print("f1 score:" , f1_score(
    results["actual_label"], results["predicted_label"], zero_division =0
))


plt.tight_layout()
plt.savefig(
    "pic3.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
results = results.copy()

def identify_scenario(row):
    flow = str(row["flow_id"]).strip().upper()
    variant = str(row["variant_id"]).strip().upper()

    for i in range(1, 6):
        if (
            flow in [f"G{i}", f"GN{i}"]
            or variant.startswith(
                (f"G{i}", f"GN{i}", f"GEN_N{i}")
            )
        ):
            return f"G{i}"

    return flow


results["scenario"] = results.apply(
    identify_scenario, axis=1
)

results["correctly_detected"] = (
    (results["actual_label"] == 1) &
    (results["predicted_label"] == 1)
).astype(int)

results["false_positive"] = (
    (results["actual_label"] == 0) &
    (results["predicted_label"] == 1)
).astype(int)

scenario_summary = (
    results.groupby("scenario")
    .agg(
        transactions=("actual_label", "size"),
        actual_anomalies=("actual_label", "sum"),
        predicted_anomalies=("predicted_label", "sum"),
        mean_score=("anomaly_score", "mean"),
        median_score=("anomaly_score", "median"),
        correctly_detected=("correctly_detected", "sum"),
        false_positives=("false_positive", "sum"),
    )
)

scenario_summary["detection_rate"] = (
    scenario_summary["correctly_detected"] /
    scenario_summary["actual_anomalies"].replace(0, np.nan)
)

scenario_summary["false_positive_rate"] = (
    scenario_summary["false_positives"] /
    (
        scenario_summary["transactions"] -
        scenario_summary["actual_anomalies"]
    ).replace(0, np.nan)
)

scenario_order = (
    [f"N{i}" for i in range(1, 6)] +
    [f"G{i}" for i in range(1, 6)] +
    [f"A{i}" for i in range(1, 6)]
)

scenario_summary = (
    scenario_summary
    .reindex(scenario_order)
    .dropna(how="all")
    .reset_index()
)

display(scenario_summary)

print("\nScenario transaction counts:")
print(results["scenario"].value_counts().reindex(scenario_order))

In [ ]:
a2 = results[
    results["scenario"] == "A2"
].copy()

a2["outcome"] = np.where(
    a2["predicted_label"] == 0,
    "MISSED",
    "DETECTED"
)

columns = [
    "transaction_id",
    "scenario",
    "length",
    "anomaly_score",
    "predicted_label",
    "outcome"
]

print("A2 transaction-level analysis")
display(
    a2[columns].sort_values("anomaly_score")
)

print("\nSummary by outcome")
display(
    a2.groupby("outcome").agg(
        transactions=("transaction_id", "count"),
        mean_length=("length", "mean"),
        mean_score=("anomaly_score", "mean"),
        min_score=("anomaly_score", "min"),
        max_score=("anomaly_score", "max")
    )
)


In [ ]:
#A2 date counts
date_counts = (
    test_df[
        test_df["sequence_token"]
        .astype(str)
        .str.contains(":date_planned:", regex=False, na=False)
    ]
    .groupby("transaction_id")
    .size()
    .rename("date_planned_count")
)

sequence_lengths = (
    test_df.groupby("transaction_id")
    .size()
    .rename("actual_length")
)

analysis = (
    results.drop(
        columns=["date_planned_count", "actual_length"],
        errors="ignore"
    )
    .merge(date_counts, on="transaction_id", how="left")
    .merge(sequence_lengths, on="transaction_id", how="left")
)

analysis["date_planned_count"] = (
    analysis["date_planned_count"].fillna(0).astype(int)
)

a2 = analysis[
    analysis["scenario"] == "A2"
].copy()

a2["outcome"] = a2["predicted_label"].map({
    1: "DETECTED",
    0: "MISSED"
})

print("\n=== ALL A2 TRANSACTIONS ===")

print(
    a2[
        [
            "transaction_id",
            "length",
            "actual_length",
            "date_planned_count",
            "anomaly_score",
            "outcome"
        ]
    ].sort_values(
        ["date_planned_count", "length"]
    ).to_string(index=False)
)

print("\n=== PLANNED-DATE EVENT SUMMARY ===")

summary = pd.crosstab(
    a2["date_planned_count"],
    a2["outcome"]
)

print(summary)

print("\n=== MISSED A2 TRANSACTIONS ===")

missed = a2[a2["predicted_label"] == 0]

print(
    missed[
        [
            "transaction_id",
            "length",
            "date_planned_count",
            "anomaly_score"
        ]
    ].to_string(index=False)
)

print("\n=== DETECTED A2 - LENGTH 12 OR 13 ===")

short_detected = a2[
    (a2["predicted_label"] == 1) &
    (a2["length"].isin([12, 13]))
]

print(
    short_detected[
        [
            "transaction_id",
            "length",
            "date_planned_count",
            "anomaly_score"
        ]
    ].to_string(index=False)
)




In [ ]:
#A2
compare_ids = ["P00193", "P00186", "P00200", "P00187"]

for tid in compare_ids:
    row = results.loc[
        results["transaction_id"] == tid
    ].iloc[0]

    events = (
        test_df[test_df["transaction_id"] == tid]
        .sort_values("sequence_no")
    )

    print("\n" + "=" * 90)
    print(
        f"{tid} | length={row['length']} | "
        f"score={row['anomaly_score']:.6f} | "
        f"prediction={row['predicted_label']}"
    )

    for _, event in events.iterrows():
        print(
            f"{int(event['sequence_no']):02d}  "
            f"{event['sequence_token']}"
        )


In [ ]:
#A2 token level recosntruction score.
compare_ids = ["P00193", "P00186", "P00200", "P00187"]
def token_errors_for_transaction(tid):
    row = test_transactions.loc[
        test_transactions["transaction_id"] == tid
    ].iloc[0]

    tokens = row["tokens"]
    token_ids = [
        token_to_idx.get(token, 1)
        for token in tokens
    ]



    encoded = pad_sequences(
        [token_ids],
        maxlen=MAXIMUM_LEN,
        padding="post",
        truncating="post",
        value=0
    )

    predictions = model.predict(encoded, verbose=0)[0]

    rows = []
    for i, token in enumerate(tokens[:MAXIMUM_LEN]):
        true_id = encoded[0, i]
        probability = np.clip(
            predictions[i, true_id],
            1e-9,
            1.0
        )

        rows.append({
            "transaction_id": tid,
            "position": i + 1,
            "token": token,
            "token_error": -np.log(probability)
        })

    return pd.DataFrame(rows)


token_error_results = pd.concat(
    [
        token_errors_for_transaction(tid)
        for tid in compare_ids
    ],
    ignore_index=True
)

for tid in compare_ids:
    print(f"\n{tid} – highest-error events")

    display(
        token_error_results[
            token_error_results["transaction_id"] == tid
        ].sort_values(
            "token_error",
            ascending=False
        ).head(8)
    )

In [ ]:
# A5
a5 = results[results["scenario"] == "A5"].copy()

display(a5)

print(a5["anomaly_score"].describe())

transaction_id = a5["transaction_id"].iloc[0]

print("\nTransaction:", transaction_id)

trace = test_df[
    test_df["transaction_id"] == transaction_id
].sort_values("sequence_no")

for _, row in trace.iterrows():
    print(f"{row['sequence_no']}: {row['sequence_token']}")

In [ ]:
#G1
g1 = results[
    results["scenario"].astype(str).str.upper() == "G1"
].copy()

if g1.empty:
    raise ValueError("No G1 transactions found in results.")

g1["outcome"] = g1["predicted_label"].map({
    0: "CORRECT",
    1: "FALSE POSITIVE"
})

print("\n=== G1 TRANSACTION RESULTS ===")

print(
    g1[
        [
            "transaction_id",
            "length",
            "anomaly_score",
            "predicted_label",
            "outcome"
        ]
    ].sort_values("transaction_id").to_string(index=False)
)

total = len(g1)
correct = (g1["predicted_label"] == 0).sum()
false_positives = (g1["predicted_label"] == 1).sum()
fpr = false_positives / total * 100

print("\n=== G1 PERFORMANCE ===")
print("Total transactions:", total)
print("Correctly classified:", correct)
print("False positives:", false_positives)
print(f"False-positive rate: {fpr:.2f}%")

print("\n=== G1 SEQUENCE CHARACTERISTICS ===")
print("Minimum length:", g1["length"].min())
print("Maximum length:", g1["length"].max())
print("Mean length:", round(g1["length"].mean(), 2))

print("Minimum anomaly score:",
      round(g1["anomaly_score"].min(), 6))
print("Maximum anomaly score:",
      round(g1["anomaly_score"].max(), 6))

example_id = g1.iloc[0]["transaction_id"]

events = (
    test_df[test_df["transaction_id"] == example_id]
    .sort_values("sequence_no")
)

print(f"\n=== G1 RUNTIME SEQUENCE: {example_id} ===")

for _, event in events.iterrows():
    print(
        f"{int(event['sequence_no']):02d} "
        f"{event['sequence_token']}"
    )


In [ ]:
#G5
g5 = results[results["scenario"] == "G5"].copy()

display(g5[[
    "transaction_id",
    "length",
    "anomaly_score",
    "predicted_label"
]])

print("\nG5 score summary:")
print(g5["anomaly_score"].describe())

transaction_id = g5["transaction_id"].iloc[0]

trace = test_df[
    test_df["transaction_id"] == transaction_id
].sort_values("sequence_no")

print("\nTransaction:", transaction_id)

for _, row in trace.iterrows():
    print(f"{row['sequence_no']}: {row['sequence_token']}")